In [5]:
get_ipython().system('unzip data.zip')

Streaming output truncated to the last 5000 lines.
  inflating: data/labels/val/vlcsnap-2025-02-19-13h52m29s173.txt  
  inflating: data/labels/val/vlcsnap-2025-02-19-13h52m41s173.txt  
  inflating: data/labels/val/vlcsnap-2025-02-19-13h52m44s869.txt  
  inflating: data/labels/val/vlcsnap-2025-02-19-13h52m55s313.txt  
  inflating: data/labels/val/vlcsnap-2025-02-19-13h52m57s218.txt  
  inflating: data/labels/val/vlcsnap-2025-02-19-13h52m59s472.txt  
  inflating: data/labels/val/vlcsnap-2025-02-19-13h53m07s405.txt  
  inflating: data/labels/val/vlcsnap-2025-02-19-13h53m09s268.txt  
  inflating: data/labels/val/vlcsnap-2025-02-19-13h53m12s243.txt  
  inflating: data/labels/val/vlcsnap-2025-02-19-13h53m21s401.txt  
  inflating: data/labels/val/vlcsnap-2025-02-19-13h53m22s735.txt  
  inflating: data/labels/val/vlcsnap-2025-02-19-13h53m26s148.txt  
  inflating: data/labels/val/vlcsnap-2025-02-19-13h53m30s157.txt  
  inflating: data/labels/val/vlcsnap-2025-02-19-13h53m34s588.txt  
  inflating

In [4]:
get_ipython().system('unzip dataset_clean.zip')

Archive:  dataset_clean.zip
  End-of-central-directory signature not found.  Either this file is not
  a zipfile, or it constitutes one disk of a multi-part archive.  In the
  latter case the central directory and zipfile comment will be found on
  the last disk(s) of this archive.
unzip:  cannot find zipfile directory in one of dataset_clean.zip or
        dataset_clean.zip.zip, and cannot find dataset_clean.zip.ZIP, period.


In [6]:
! pip install ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.7/46.7 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 37.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.0/90.0 kB 8.9 MB/s eta 0:00:00


In [ ]:
"""
=====================================================
YOLOv11 - Road Defect Detection (v7) — FULL PIPELINE (FIXED)
=====================================================
แก้ไขจากรอบตรวจสอบ syntax/logic:
1. FIX: oversample_crack_images() เดิม copy path ซ้ำแล้วถูกทับกันในปลายทาง
   -> copy_split_to_workdir() ตอนนี้ตั้งชื่อไฟล์ใหม่ไม่ให้ชนกันเมื่อเจอไฟล์ซ้ำ
2. FIX: CLAHE เดิมทำเฉพาะ train แต่ val/test เป็นภาพดิบ (domain mismatch)
   -> ตอนนี้ CLAHE ใช้กับทุก split เท่ากันถ้าเปิดใช้งาน
   -> ปิดเป็นค่าเริ่มต้น (USE_CLAHE=False) ให้ทดสอบทีละตัวแปรก่อน
3. เพิ่มการรายงานสัดส่วนภาพจริงหลัง split (ไม่ใช่แค่จำนวนกลุ่ม)
=====================================================
"""

import os
import re
import glob
import random
import shutil
import numpy as np
import cv2
from pathlib import Path
from datetime import datetime
from collections import defaultdict
from ultralytics import YOLO

# =====================================================
# 1. CONFIG
# =====================================================
CONFIG = {
    "model": "yolo11m.pt",
    "optimizer": "AdamW",
    "lr0": 0.0015,
    "lrf": 0.01,
    "momentum": 0.937,
    "weight_decay": 0.0005,
    "dropout": 0.0,
    "box": 7.5,
    "cls": 0.6,
    "dfl": 1.5,
    "epochs": 100,
    "patience": 30,
    "imgsz": 1280,
    "batch": 0.8,
    "workers": 6,
    "deterministic": True,
    "seed": 0,
    "amp": True,
    "device": 0,
    "mosaic": 1.0,
    "close_mosaic": 15,
    "mixup": 0.0,
    "copy_paste": 0.0,
    "degrees": 0.0,
    "translate": 0.1,
    "scale": 0.5,
    "hsv_h": 0.015,
    "hsv_s": 0.7,
    "hsv_v": 0.4,
}

VAL_CONF_THRESHOLD = 0.001
VAL_IOU_THRESHOLD = 0.6
DEPLOY_CONF_THRESHOLD = 0.25

# --- PATH ที่ถูกต้อง ตามโครงสร้างไฟล์จริง ---
SRC_DIR = "/content/data"                  # ข้อมูลดิบ: data/images, data/labels
WORK_DIR = "/content/dataset_clean"        # ผลลัพธ์หลัง split: images/{train,val,test}

IMAGES_GLOB = os.path.join(SRC_DIR, "images", "*.*")
CLASSES = ["pothole", "crack", "manhole"]
CRACK_CLASS_ID = 1

TIME_GAP_SEC = 90
FRAME_GAP = 5
MAX_GROUP_SIZE = 25

SPLIT_RATIOS = (0.75, 0.15, 0.10)  # train / val / test
SEED = 0

# FIX #2: ปิด CLAHE เป็นค่าเริ่มต้น เพื่อทดสอบทีละตัวแปรก่อน
# ถ้าเปิด (True) จะใช้กับ train/val/test เท่ากันหมด ไม่ทำแค่ train เหมือนเดิม
USE_CLAHE = False
CLAHE_OUTPUT_DIR = os.path.join(WORK_DIR, "images_clahe_tmp")
CRACK_OVERSAMPLE_RATIO = 1.5


# =====================================================
# 2. METADATA EXTRACTION (logic จริงจาก v6)
# =====================================================
def extract_timestamp(filename):
    """
    ดึง timestamp จากชื่อไฟล์ รองรับ 2 pattern:
    1) vlcsnap_YYYY-MM-DD-HHhMMmSSs### (screenshot จาก VLC)
    2) YYYYMMDD_HHMMSS
    คืนค่า datetime object หรือ None ถ้าไม่ match
    """
    stem = Path(filename).stem

    pattern_vlc = r"^vlcsnap[_-](\d{4})-(\d{2})-(\d{2})-(\d{2})h(\d{2})m(\d{2})s\d*$"
    m = re.match(pattern_vlc, stem)
    if m:
        year, month, day, hour, minute, second = map(int, m.groups())
        return datetime(year, month, day, hour, minute, second)

    pattern_std = r"^(\d{8})_(\d{6})$"
    m = re.match(pattern_std, stem)
    if m:
        date_part, time_part = m.groups()
        return datetime.strptime(date_part + time_part, "%Y%m%d%H%M%S")

    return None


def extract_frame_number(filename):
    """
    ดึง prefix + เลขเฟรมจาก pattern: prefix-number(-suffix)?
    เช่น "road-00123" หรือ "road-00123-aug"
    คืนค่า (prefix, frame_number) หรือ None
    """
    stem = Path(filename).stem
    pattern = r"^([A-Za-z]+)-(\d+)(?:-[A-Za-z0-9]+)?$"
    m = re.match(pattern, stem)
    if m:
        prefix, num_str = m.groups()
        return prefix, int(num_str)
    return None


def is_same_group_time(ts, prev_ts, time_gap_sec=TIME_GAP_SEC):
    return (ts - prev_ts).total_seconds() <= time_gap_sec


def is_same_group_frame(prefix, num, prev_prefix, prev_num, frame_gap=FRAME_GAP):
    return prefix == prev_prefix and (num - prev_num) <= frame_gap


# =====================================================
# 3. BUILD_GROUPS (logic จริงจาก v6, คืนค่าเป็น final_map)
# =====================================================
def build_groups(img_paths, time_gap_sec=TIME_GAP_SEC,
                  frame_gap=FRAME_GAP, max_group_size=MAX_GROUP_SIZE):
    """
    จัดกลุ่มภาพที่มาจากช่วงเวลา/เฟรมต่อเนื่องกันให้อยู่กลุ่มเดียวกัน
    เพื่อป้องกัน data leakage ระหว่าง train/val/test

    คืนค่า: final_map -> dict {img_path: group_name}
    """
    with_time, with_frame, others = [], [], []

    for p in img_paths:
        fname = os.path.basename(p)
        ts = extract_timestamp(fname)
        if ts is not None:
            with_time.append((p, ts))
            continue
        fr = extract_frame_number(fname)
        if fr is not None:
            prefix, num = fr
            with_frame.append((p, prefix, num))
            continue
        others.append(p)

    raw_groups = defaultdict(list)

    # --- Time-based grouping ---
    with_time.sort(key=lambda x: x[1])
    gid = 0
    prev_ts = None
    for p, ts in with_time:
        if prev_ts is not None and not is_same_group_time(ts, prev_ts, time_gap_sec):
            gid += 1
        raw_groups[f"time_{gid}"].append(p)
        prev_ts = ts

    # --- Frame-based grouping ---
    with_frame.sort(key=lambda x: (x[1], x[2]))
    gid = 0
    prev_prefix, prev_num = None, None
    for p, prefix, num in with_frame:
        if prev_prefix is not None and not is_same_group_frame(
            prefix, num, prev_prefix, prev_num, frame_gap
        ):
            gid += 1
        raw_groups[f"frame_{gid}"].append(p)
        prev_prefix, prev_num = prefix, num

    # --- Others: แต่ละไฟล์เป็นกลุ่มเดี่ยว ---
    for i, p in enumerate(others):
        raw_groups[f"single_{i}"].append(p)

    # --- บังคับ MAX_GROUP_SIZE: ตัดกลุ่มใหญ่เกินเป็น chunk ---
    final_map = {}
    for gname, paths in raw_groups.items():
        if len(paths) <= max_group_size:
            for p in paths:
                final_map[p] = gname
        else:
            for i in range(0, len(paths), max_group_size):
                chunk_name = f"{gname}_chunk{i // max_group_size}"
                for p in paths[i:i + max_group_size]:
                    final_map[p] = chunk_name

    return final_map


# =====================================================
# 4. STRATIFIED GROUP SPLIT (ใหม่ใน v7)
# =====================================================
def label_path(img_p, src_dir=SRC_DIR):
    """ดึง path label .txt ที่ตรงกับไฟล์ภาพ"""
    base = os.path.splitext(os.path.basename(img_p))[0]
    return os.path.join(src_dir, "labels", base + ".txt")


def count_crack_instances(img_paths, src_dir=SRC_DIR):
    """นับจำนวน crack instance จาก label ของแต่ละไฟล์ภาพ"""
    total = 0
    for img_p in img_paths:
        lp = label_path(img_p, src_dir)
        if os.path.exists(lp):
            with open(lp) as fp:
                total += sum(1 for line in fp if line.strip().startswith(str(CRACK_CLASS_ID)))
    return total


def group_aware_split(final_map, ratios=SPLIT_RATIOS, seed=SEED):
    """
    รับ final_map (dict: img_path -> group_name) จาก build_groups
    แปลงกลับเป็น list-of-groups แล้ว stratify ตามจำนวน crack instance
    เพื่อให้ทุก split มีสัดส่วน crack ใกล้เคียงกัน
    """
    random.seed(seed)

    grouped = defaultdict(list)
    for path, gname in final_map.items():
        grouped[gname].append(path)

    groups_list = list(grouped.values())
    groups_sorted = sorted(groups_list, key=count_crack_instances, reverse=True)

    train, val, test = [], [], []
    buckets = [train, val, test]
    counters = [0.0, 0.0, 0.0]

    for g in groups_sorted:
        total_so_far = sum(counters) + 1
        deficit = [ratios[i] - (counters[i] / total_so_far) for i in range(3)]
        idx = int(np.argmax(deficit))
        buckets[idx].append(g)
        counters[idx] += 1

    def flatten(bs):
        return [f for g in bs for f in g]

    return flatten(train), flatten(val), flatten(test)


# =====================================================
# 5. CRACK-FOCUSED PREPROCESSING
# =====================================================
def apply_clahe_to_dataset(image_paths, output_dir):
    """
    เพิ่มคอนทราสต์เฉพาะจุดด้วย CLAHE เพื่อให้เส้น crack เด่นขึ้น
    FIX #2: ต้องเรียกกับทุก split (train/val/test) เท่ากัน ไม่ใช่แค่ train
    เพื่อไม่ให้เกิด domain mismatch ระหว่างตอนเทรนกับตอนวัดผล
    """
    os.makedirs(output_dir, exist_ok=True)
    clahe = cv2.createCLAHE(clipLimit=2.5, tileGridSize=(8, 8))
    processed = []
    for img_path in image_paths:
        img = cv2.imread(str(img_path))
        if img is None:
            continue
        lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
        l, a, b = cv2.split(lab)
        l2 = clahe.apply(l)
        enhanced = cv2.merge((l2, a, b))
        enhanced = cv2.cvtColor(enhanced, cv2.COLOR_LAB2BGR)
        out_path = Path(output_dir) / Path(img_path).name
        cv2.imwrite(str(out_path), enhanced)
        processed.append(str(out_path))
    return processed


def oversample_crack_images(train_files, target_ratio=CRACK_OVERSAMPLE_RATIO, src_dir=SRC_DIR):
    """
    เพิ่มจำนวนภาพที่มี crack ใน train set เท่านั้น (ไม่แตะ val/test)
    หมายเหตุ: list ที่คืนกลับมาจะมี path ซ้ำกันได้ตั้งใจ
    -> copy_split_to_workdir() ต้องจัดการ rename ไม่ให้ไฟล์ทับกัน (ดู FIX #1)
    """
    crack_files = [f for f in train_files if count_crack_instances([f], src_dir) > 0]
    n_extra = int(len(crack_files) * (target_ratio - 1.0))
    extra = random.sample(crack_files, min(n_extra, len(crack_files))) if crack_files else []
    return train_files + extra


# =====================================================
# 6. COPY SPLIT TO WORKDIR (FIX #1: กัน path ซ้ำถูกทับกัน)
# =====================================================
def copy_split_to_workdir(train_files, val_files, test_files,
                           src_dir=SRC_DIR, work_dir=WORK_DIR,
                           clear_existing=True):
    """
    คัดลอกไฟล์ภาพ + label ตาม split ไปยัง work_dir/images/{train,val,test}
    และ work_dir/labels/{train,val,test}

    FIX #1: เดิมถ้า train_files มี path ซ้ำกัน (จาก oversample_crack_images)
    การ copy2 ไปที่ไฟล์ชื่อเดิมซ้ำๆ จะโดนทับกันเหลือไฟล์เดียว ทำให้ oversample
    ไม่มีผลจริงตอนเทรน -> ตอนนี้ถ้าเจอไฟล์ชื่อซ้ำ จะตั้งชื่อใหม่ไม่ให้ชนกัน
    (เช่น road-001.jpg, road-001_dup2.jpg)
    """
    splits = {"train": train_files, "val": val_files, "test": test_files}

    for split_name in splits:
        img_dst_dir = Path(work_dir) / "images" / split_name
        lbl_dst_dir = Path(work_dir) / "labels" / split_name

        if clear_existing and img_dst_dir.exists():
            shutil.rmtree(img_dst_dir)
        if clear_existing and lbl_dst_dir.exists():
            shutil.rmtree(lbl_dst_dir)

        img_dst_dir.mkdir(parents=True, exist_ok=True)
        lbl_dst_dir.mkdir(parents=True, exist_ok=True)

    summary = {}
    missing_labels = []

    for split_name, file_list in splits.items():
        img_dst_dir = Path(work_dir) / "images" / split_name
        lbl_dst_dir = Path(work_dir) / "labels" / split_name

        copied_img, copied_lbl, skipped = 0, 0, 0
        name_counter = {}  # FIX #1: นับว่าไฟล์ชื่อนี้เจอกี่ครั้งแล้วใน split นี้

        for img_p in file_list:
            img_p = Path(img_p)
            if not img_p.exists():
                skipped += 1
                continue

            # FIX #1: ตั้งชื่อไฟล์ใหม่ถ้าเจอซ้ำ ป้องกันการทับกัน
            name_counter[img_p.name] = name_counter.get(img_p.name, 0) + 1
            occurrence = name_counter[img_p.name]
            if occurrence == 1:
                dst_img_name = img_p.name
            else:
                dst_img_name = f"{img_p.stem}_dup{occurrence}{img_p.suffix}"

            shutil.copy2(img_p, img_dst_dir / dst_img_name)
            copied_img += 1

            lbl_src = Path(src_dir) / "labels" / (img_p.stem + ".txt")
            if lbl_src.exists():
                dst_lbl_name = Path(dst_img_name).stem + ".txt"
                shutil.copy2(lbl_src, lbl_dst_dir / dst_lbl_name)
                copied_lbl += 1
            else:
                missing_labels.append(str(img_p))

        summary[split_name] = {
            "total_requested": len(file_list),
            "unique_files": len(name_counter),
            "images_copied": copied_img,
            "labels_copied": copied_lbl,
            "images_skipped_not_found": skipped,
        }

    print("=== สรุปการคัดลอกไฟล์ตาม split ===")
    for split_name, stats in summary.items():
        print(f"[{split_name.upper()}] "
              f"ขอ: {stats['total_requested']} (ไม่ซ้ำ: {stats['unique_files']}) | "
              f"คัดลอกภาพสำเร็จ: {stats['images_copied']} | "
              f"คัดลอก label สำเร็จ: {stats['labels_copied']} | "
              f"ไฟล์ที่หาไม่เจอ: {stats['images_skipped_not_found']}")

    if missing_labels:
        print(f"\nหมายเหตุ: มี {len(missing_labels)} ภาพไม่มี label คู่กัน "
              f"(background image ปกติของ YOLO)")

    return summary


def update_data_yaml(work_dir=WORK_DIR, classes=CLASSES):
    """เขียน data.yaml ให้ชี้ไปที่โฟลเดอร์ images/{train,val,test}"""
    yaml_path = Path(work_dir) / "data.yaml"
    content = f"""train: {work_dir}/images/train
val: {work_dir}/images/val
test: {work_dir}/images/test
nc: {len(classes)}
names: {classes}
"""
    with open(yaml_path, "w") as f:
        f.write(content)

    print(f"\nอัปเดต data.yaml แล้วที่: {yaml_path}")
    print(content)
    return str(yaml_path)


# =====================================================
# 7. TRAIN
# =====================================================
def train_model(data_yaml_path):
    model = YOLO(CONFIG["model"])
    results = model.train(
        data=data_yaml_path,
        epochs=CONFIG["epochs"],
        patience=CONFIG["patience"],
        imgsz=CONFIG["imgsz"],
        batch=CONFIG["batch"],
        optimizer=CONFIG["optimizer"],
        lr0=CONFIG["lr0"],
        lrf=CONFIG["lrf"],
        momentum=CONFIG["momentum"],
        weight_decay=CONFIG["weight_decay"],
        dropout=CONFIG["dropout"],
        box=CONFIG["box"],
        cls=CONFIG["cls"],
        dfl=CONFIG["dfl"],
        mosaic=CONFIG["mosaic"],
        close_mosaic=CONFIG["close_mosaic"],
        mixup=CONFIG["mixup"],
        copy_paste=CONFIG["copy_paste"],
        degrees=CONFIG["degrees"],
        translate=CONFIG["translate"],
        scale=CONFIG["scale"],
        hsv_h=CONFIG["hsv_h"],
        hsv_s=CONFIG["hsv_s"],
        hsv_v=CONFIG["hsv_v"],
        device=CONFIG["device"],
        workers=CONFIG["workers"],
        seed=CONFIG["seed"],
        deterministic=CONFIG["deterministic"],
        amp=CONFIG["amp"],
        verbose=True,
    )
    return model, results


# =====================================================
# 8. VALIDATE (มาตรฐานเท่านั้น)
# =====================================================
def evaluate_model(model, data_yaml_path, split="val"):
    metrics = model.val(
        data=data_yaml_path,
        split=split,
        conf=VAL_CONF_THRESHOLD,
        iou=VAL_IOU_THRESHOLD,
        imgsz=CONFIG["imgsz"],
        verbose=True,
    )
    print(f"\n{split.upper()}  mAP50: {metrics.box.map50:.4f} | mAP50-95: {metrics.box.map:.4f}")
    for i, cls_name in enumerate(CLASSES):
        p = metrics.box.p[i] if len(metrics.box.p) > i else 0
        r = metrics.box.r[i] if len(metrics.box.r) > i else 0
        ap50 = metrics.box.ap50[i] if len(metrics.box.ap50) > i else 0
        print(f"  {cls_name:10s}  P={p:.3f}  R={r:.3f}  mAP50={ap50:.3f}")
    return metrics


# =====================================================
# 9. MAIN PIPELINE
# =====================================================
if __name__ == "__main__":
    print("=== V7 Training Pipeline (full, fixed) ===")
    print("1. Group-aware split (v6 logic) + stratify ตาม crack instance")
    print("2. Copy ไฟล์จริงไปยัง dataset_clean (กัน oversample ถูกทับกัน)")
    print("3. Oversample crack เฉพาะ train set")
    print(f"4. CLAHE preprocessing: {'เปิดใช้งาน (ทุก split)' if USE_CLAHE else 'ปิดอยู่ (ทดสอบทีละตัวแปร)'}")
    print("5. Train ด้วย yolo11m, imgsz=1280, cls=0.6, patience=30")
    print("6. Validate มาตรฐาน conf=0.001, iou=0.6\n")

    # --- Step 1: scan ไฟล์ภาพ ---
    img_paths = sorted(glob.glob(IMAGES_GLOB))
    print(f"จำนวนไฟล์ภาพที่ scan เจอ: {len(img_paths)}")
    if len(img_paths) == 0:
        raise FileNotFoundError(
            f"ไม่พบไฟล์ภาพใน {IMAGES_GLOB} — ตรวจสอบ SRC_DIR ว่ามีโฟลเดอร์ images/ อยู่จริง"
        )

    # --- Step 2: จัดกลุ่มแบบ group-aware (logic จริงจาก v6) ---
    final_map = build_groups(img_paths, TIME_GAP_SEC, FRAME_GAP, MAX_GROUP_SIZE)
    n_groups = len(set(final_map.values()))
    print(f"จำนวนกลุ่มทั้งหมด: {n_groups}")

    # --- Step 3: stratified split ตาม crack instance ---
    train_files, val_files, test_files = group_aware_split(final_map)
    total_imgs = len(train_files) + len(val_files) + len(test_files)
    print(f"ก่อน oversample -> Train: {len(train_files)} ({len(train_files)/total_imgs:.1%}) | "
          f"Val: {len(val_files)} ({len(val_files)/total_imgs:.1%}) | "
          f"Test: {len(test_files)} ({len(test_files)/total_imgs:.1%})")

    train_crack_n = count_crack_instances(train_files)
    val_crack_n = count_crack_instances(val_files)
    test_crack_n = count_crack_instances(test_files)
    total_crack = train_crack_n + val_crack_n + test_crack_n
    print(f"Crack instances -> Train: {train_crack_n} ({train_crack_n/max(total_crack,1):.1%}) | "
          f"Val: {val_crack_n} ({val_crack_n/max(total_crack,1):.1%}) | "
          f"Test: {test_crack_n} ({test_crack_n/max(total_crack,1):.1%})")

    # --- Step 4: oversample crack เฉพาะ train ---
    train_files = oversample_crack_images(train_files)
    print(f"หลัง oversample -> Train: {len(train_files)} (มีไฟล์ซ้ำโดยตั้งใจ)")

    # --- Step 5: (ตัวเลือก) CLAHE preprocessing ให้ทุก split เท่ากัน ---
    if USE_CLAHE:
        print("\nกำลังทำ CLAHE preprocessing (train/val/test เท่ากันหมด)...")
        train_files = apply_clahe_to_dataset(train_files, os.path.join(CLAHE_OUTPUT_DIR, "train"))
        val_files = apply_clahe_to_dataset(val_files, os.path.join(CLAHE_OUTPUT_DIR, "val"))
        test_files = apply_clahe_to_dataset(test_files, os.path.join(CLAHE_OUTPUT_DIR, "test"))

    # --- Step 6: copy ไฟล์จริงไปยัง dataset_clean ---
    copy_split_to_workdir(train_files, val_files, test_files)
    data_yaml_path = update_data_yaml()

    # --- Step 7: เทรน ---
    model, train_results = train_model(data_yaml_path)

    # --- Step 8: validate ---
    print("\n--- Validation Set ---")
    val_metrics = evaluate_model(model, data_yaml_path, split="val")

    print("\n--- Test Set ---")
    test_metrics = evaluate_model(model, data_yaml_path, split="test")

    # --- Step 9: เทียบกับ v6 ---
    print("\n=== เทียบผลลัพธ์ v6 vs v7 ===")
    print("v6  VAL mAP50: 0.4584 | TEST mAP50: 0.4339")
    print(f"v7  VAL mAP50: {val_metrics.box.map50:.4f} | TEST mAP50: {test_metrics.box.map50:.4f}")

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
=== V7 Training Pipeline (full, fixed) ===
1. Group-aware split (v6 logic) + stratify ตาม crack instance
2. Copy ไฟล์จริงไปยัง dataset_clean (กัน oversample ถูกทับกัน)
3. Oversample crack เฉพาะ train set
4. CLAHE preprocessing: ปิดอยู่ (ทดสอบทีละตัวแปร)
5. Train ด้วย yolo11m, imgsz=1280, cls=0.6, patience=30
6. Validate มาตรฐาน conf=0.001, iou=0.6

จำนวนไฟล์ภาพที่ scan เจอ: 2009
จำนวนกลุ่มทั้งหมด: 99
ก่อน oversample -> Train: 1466 (73.0%) | Val: 311 (15.5%) | Test: 232 (11.5%)
Crack instances -> Train: 1891 (75.1%) | Val: 382 (15.2%) | Test: 246 (9.8%)
หลัง oversample -> Train: 1988 (มีไฟล์ซ้ำโดยตั้งใจ)
=== สรุปการคัดลอกไฟล์ตาม split ===
[TRAIN] ขอ: 1988 (ไม่ซ้ำ: 1466) | คัดลอกภาพสำเร็จ: 1988 | คัด